# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row in fact_content_daily_performance is one content item, for one client, on one day. Development window is January and February 2026 (feature period). March 2026 is the label period, used only to observe what actually happened after the Jan-Feb decision point. April through June 2026 is untouched, since that range overlaps both the dataset's fixed fact_content_query_90d snapshot and the sealed _sample test month.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields: gsc_clicks, gsc_impressions, gsc_avg_position, sessions_ai, ga4_sessions, all read only from Jan-Feb and aggregated per content item into avg_daily_clicks, avg_daily_impressions, avg_position, ai_sessions_share, early_late_click_delta. Label field: gsc_clicks from March, aggregated into label_window_avg_clicks, thresholded into is_declining (March average below 80% of Jan-Feb average). Context fields: client_hash_id, content_hash_id, report_date, used as keys and filters, never as model inputs. Excluded fields: every other traffic-source and engagement column in the table (ga4_pageviews, sessions_direct/referral/social/paid, individual ai_chatgpt/perplexity/gemini/... breakdowns, scroll_events), excluded because this lane scopes to GSC search performance specifically, and client_has_gsc/gsc_data_available, used only as row filters, excluded from the feature set since they're eligibility flags, not signal.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three query cells back this contract: a grouping query on (report_date, client_hash_id, content_hash_id) confirming max group size is 1, proving the stated grain; a count and date-span query on Jan-Feb filtered to client_has_gsc IS TRUE, giving slice size and confirming the window boundaries; an availability query adding gsc_data_available IS TRUE, showing how many of those rows survive and quantifying the missingness the contract excludes.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Honest AUC on the five features came out to 0.58, barely above chance, meaning this feature set carries weak standalone signal for decline. Class balance is uneven at roughly 67/33 not-declining/declining. GSC availability is not randomly distributed across clients, so this slice reflects only clients who connected GSC, not the full content population. Content items newer than the Jan-Feb window have no prior history to average over and get dropped by the feature-window aggregation, biasing the analyzed set toward older, established content.